In [ ]:
# V4 버전

import pandas as pd
import random

# =====================================
# 학생 데이터 불러오기
# =====================================

students = pd.read_csv("../data/students.csv")

dates = pd.date_range(
    start="2026-07-01",
    periods=30
)

In [2]:
# =====================================
# 학생 성향 배정
# =====================================

profiles = (
    ["최상위권"] * 3 +
    ["성장형"] * 5 +
    ["안정형"] * 6 +
    ["기복형"] * 4 +
    ["슬럼프형"] * 2
)

random.shuffle(profiles)

student_status = pd.DataFrame({
    "student_id": students["student_id"],
    "profile": profiles
})

In [3]:
# =====================================
# 학생 능력치 설정
# =====================================

targets = {
    "최상위권": 98,
    "성장형": 90,
    "안정형": 84,
    "기복형": 85,
    "슬럼프형": 78
}

growth_rates = {
    "최상위권": 0.05,
    "성장형": 0.18,
    "안정형": 0.08,
    "기복형": 0.10,
    "슬럼프형": -0.05
}

advance_gap = {
    "최상위권": 1,
    "성장형": 3,
    "안정형": 4,
    "기복형": 5,
    "슬럼프형": 6
}

In [4]:
# =====================================
# 초기 점수 생성
# =====================================

current_scores = []

for profile in student_status["profile"]:

    if profile == "최상위권":
        current_scores.append(random.randint(93,96))

    elif profile == "성장형":
        current_scores.append(random.randint(55,65))

    elif profile == "안정형":
        current_scores.append(random.randint(78,83))

    elif profile == "기복형":
        current_scores.append(random.randint(72,84))

    elif profile == "슬럼프형":
        current_scores.append(random.randint(82,90))

student_status["current_score"] = current_scores
student_status["target"] = student_status["profile"].map(targets)
student_status["growth_rate"] = student_status["profile"].map(growth_rates)
student_status["advance_gap"] = student_status["profile"].map(advance_gap)

In [5]:
# =====================================
# 날짜별 시험 난이도
# =====================================

difficulty = {}

for date in dates:
    difficulty[date.strftime("%Y-%m-%d")] = random.randint(-3,3)

In [6]:
# =====================================
# DT 데이터 생성
# =====================================

records = []

for _, student in student_status.iterrows():

    current = student["current_score"]
    target = student["target"]
    growth_rate = student["growth_rate"]
    gap = student["advance_gap"]
    profile = student["profile"]

    for date in dates:

        today = date.strftime("%Y-%m-%d")

        # 성장량
        growth = (target - current) * growth_rate

        # 컨디션
        condition = random.randint(-2,2)

        # 시험 난이도
        exam = difficulty[today]

        # 기복형은 변동폭을 조금 더 크게
        if profile == "기복형":
            condition += random.randint(-3,3)

        # 현재 실력 업데이트
        current = current + growth + condition - exam

        current = round(current)

        current = max(0,min(current,100))

        # 현행 DT
        current_dt = current

        # 선행 DT
        advanced_dt = current_dt - gap + random.randint(-1,1)

        advanced_dt = max(0,min(advanced_dt,100))

        # 저장
        records.append([
            student["student_id"],
            today,
            "현행",
            current_dt
        ])

        records.append([
            student["student_id"],
            today,
            "선행",
            advanced_dt
        ])

In [7]:

# =====================================
# DataFrame 생성
# =====================================

dt_scores = pd.DataFrame(
    records,
    columns=[
        "student_id",
        "date",
        "dt_type",
        "score"
    ]
)

In [8]:

# =====================================
# CSV 저장
# =====================================

dt_scores.to_csv(
    "../data/dt_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

In [9]:

# =====================================
# 결과 확인
# =====================================

print(student_status)

print()

print(dt_scores.head())

print()

print(dt_scores.tail())

   student_id profile  current_score  target  growth_rate  advance_gap
0     4MR_001     안정형             79      84         0.08            4
1     4MR_002     기복형             82      85         0.10            5
2     4MR_003     안정형             79      84         0.08            4
3     4MR_004     안정형             78      84         0.08            4
4     4MR_005    슬럼프형             83      78        -0.05            6
5    6MR1_001     기복형             84      85         0.10            5
6    6MR1_002     기복형             75      85         0.10            5
7    6MR1_003    최상위권             96      98         0.05            1
8    6MR1_004     성장형             65      90         0.18            3
9    6MR1_005     성장형             59      90         0.18            3
10   7MS1_001    최상위권             94      98         0.05            1
11   7MS1_002     성장형             56      90         0.18            3
12   7MS1_003     안정형             80      84         0.08            4
13   7